# Example: plotting the results

Standalone example of how to use the data produced by `main.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

experiment_folder = "experiment"


In [ ]:
# Example plotting
# ##################

# Load saved results if starting from an existing experiment folder
scenarios = pd.read_parquet(Path(experiment_folder) / "scenarios.parquet")
df = pd.read_parquet(Path(experiment_folder) / "output.parquet")

d = df.dropna().copy()  # remove NA values
d = d[d["survey"] != 1]  # remove first survey (due to accumulating data during the burn-in period)
d = d.groupby(["index", "measure", "survey"], as_index=False)["value"].sum()  # aggregate age-groups

# merge with the scenarios table to have all the metadata (eir, access, seed, etc.) in the same dataframe
d = d.merge(scenarios, on="index")
d = d.sort_values(["eir", "seed", "survey", "measure"])

nPatent = d[d["measure"] == 3].copy()
nUncomp = d[d["measure"] == 14].copy()
nHost = d[d["measure"] == 0].copy()
inputEIR = d[d["measure"] == 35].copy()
simulatedEIR = d[d["measure"] == 36].copy()

# align by scenario/survey rather than relying on row order
keys = ["index", "survey"]
host = nHost[keys + ["value"]].rename(columns={"value": "host"})

prevalence = nPatent.merge(host, on=keys)
prevalence["value"] = prevalence["value"] / prevalence["host"]

incidence = nUncomp.merge(host, on=keys)
incidence["value"] = incidence["value"] / incidence["host"]

eir = pd.concat([
    inputEIR.assign(type="input"),
    simulatedEIR.assign(type="simulated"),
])

fig, axes = plt.subplots(3, 1, figsize=(9, 12))

for eir_value, group in prevalence.groupby("eir"):
    mean = group.groupby("survey", as_index=False)["value"].mean()  # stat_summary(fun = mean)
    axes[0].plot(mean["survey"], mean["value"], label=str(eir_value))
axes[0].set(xlabel="Survey", ylabel="Prevalence")
axes[0].legend(title="EIR")

for eir_value, group in incidence.groupby("eir"):
    mean = group.groupby("survey", as_index=False)["value"].mean()
    axes[1].plot(mean["survey"], mean["value"], label=str(eir_value))
axes[1].set(xlabel="Survey", ylabel="Clinical Incidence")
axes[1].legend(title="EIR")

for (eir_value, kind), group in eir.groupby(["eir", "type"]):
    mean = group.groupby("survey", as_index=False)["value"].mean()
    axes[2].plot(
        mean["survey"], mean["value"],
        linestyle="--" if kind == "input" else "-",
        label=f"{eir_value} ({kind})",
    )
axes[2].set(xlabel="Survey", ylabel="EIR")
axes[2].legend(title="EIR")

plt.tight_layout()
plt.show()
